# Direction 2 · Phase C · Step 7b: Dual-GPU (`2x T4`) Spawrious (`o2o_hard`) Training & Val Controls

**Settings:** Enable **GPU T4 x2** and **Internet ON**.

**Safe to run in parallel with Step 6b** (zero dependency on Step 6b & **zero OOD test access**):
- Clones `spurious_correlation_mechanisms` from GitHub.
- Streams `spawrious224__o2o_hard.tar.gz` (~7.03 GB) directly into `/tmp/step7b_spawrious_assets` so raw images do not exhaust Kaggle's 20 GB `/kaggle/working` quota.
- Strictly excludes all 4 shifted OOD test combinations (`mountain/bulldog`, `snow/dachshund`, `desert/labrador`, `jungle/corgi`), splitting official training domains `0` & `1` stratified by the 8 `(class x spur/beach)` groups into `85% train` / `15% val`.
- Dispatches seeds `[0, 1, 2]` across **both T4 GPUs (`cuda:0` and `cuda:1`) in parallel** to train 4-class ResNet-50 standard-ERM backbones (`15` epochs, AMP fp16) and run the 5-fold validation control ladder (`erm_raw`, `dfr_l2`, `dar`, `affine_dfr_bn_l3`, `affine_dfr_bn_l4`, `affine_dfr_bn_all`).
- Saves checkpoints in `/kaggle/working/spawrious_o2o_hard_backbones/` and zips compact diagnostics to `/kaggle/working/rep_repair_step7b_spawrious_diag.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/tmp/step7b_spawrious_assets')
CKPT_DIR = Path('/kaggle/working/spawrious_o2o_hard_backbones')
OUT_DIR = Path('/kaggle/working/rep_repair_step7b_spawrious_diag')

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_names = [torch.cuda.get_device_name(i) for i in range(n_gpus)]
print(f'CUDA available: {torch.cuda.is_available()} | GPUs ({n_gpus}): {gpu_names}')

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Pre-flight unit tests.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_step7b_spawrious.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_step7b_spawrious',
    '--work-dir', str(WORK_DIR),
    '--ckpt-dir', str(CKPT_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--max-gpus', '2',
    '--workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'step7b_spawrious_controls.png')))

print('Saved Spawrious o2o_hard checkpoints (retained in /kaggle/working/spawrious_o2o_hard_backbones):')
for p in sorted(CKPT_DIR.glob('*.pt')):
    print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

zip_path = Path('/kaggle/working/rep_repair_step7b_spawrious_diag.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved diagnostic bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')